## Setup

In [1]:
# Load the standard libraries and the API key from .env
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from anthropic.types import Message

# Create the client and choose the model
client = Anthropic()
model = "claude-haiku-4-5"

## Helper functions

In [2]:
# Add a user turn; accepts plain text or a full Message object
def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })

# Add an assistant turn; keeps ALL blocks (thinking, text) intact
def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })

# Chat function with optional extended thinking
def chat(messages, system=None, temperature=1.0, stop_sequences=[],
         tools=None, thinking=False, thinking_budget=1024):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    # Thinking requires default temperature, so only set it when thinking is off
    if not thinking:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system

    if thinking:
        # max_tokens must be larger than the thinking budget
        params["max_tokens"] = thinking_budget + 1000
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,  # minimum is 1024
        }

    # Return the whole message, not just the text
    return client.messages.create(**params)

## Reading the response blocks

In [3]:
# Pull out only the final answer text
def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

# Show every block, including the reasoning
def show_blocks(message):
    for block in message.content:
        if block.type == "thinking":
            print("THINKING:", block.thinking[:300], "...")
            print("SIGNATURE (first 30 chars):", block.signature[:30])
        elif block.type == "redacted_thinking":
            print("REDACTED THINKING (encrypted, not readable)")
        elif block.type == "text":
            print("ANSWER:", block.text)
        print("-" * 40)

## Test without thinking

In [4]:
# A reasoning question to use for the comparison
question = ("A factory machine makes 120 parts per hour and 4% are defective. "
            "How many good parts are made in a 7.5 hour shift?")

messages = []
add_user_message(messages, question)

# Baseline: no extended thinking
response = chat(messages)
print(text_from_message(response))

# Calculating Good Parts Made

**Step 1: Find total parts made in 7.5 hours**
- 120 parts/hour × 7.5 hours = 900 parts

**Step 2: Find the percentage of good parts**
- If 4% are defective, then 96% are good
- 100% - 4% = 96%

**Step 3: Calculate good parts**
- 900 × 0.96 = **864 good parts**


## Test with thinking

In [5]:
messages = []
add_user_message(messages, question)

# Same question, with extended thinking enabled
response = chat(messages, thinking=True)
show_blocks(response)

# Token usage shows the extra cost of thinking
print(response.usage)

THINKING: Let me work through this step by step.

The machine makes 120 parts per hour.
4% are defective, which means 96% are good (100% - 4% = 96%).

In a 7.5 hour shift:
Total parts made = 120 parts/hour × 7.5 hours = 900 parts

Good parts = 96% of 900 parts
Good parts = 0.96 × 900 = 864 parts

Alternativel ...
SIGNATURE (first 30 chars): EsYFCrgBCBMYAipA73T/a9xXpFPwCl
----------------------------------------
ANSWER: # Good Parts Calculation

**Total parts made in 7.5 hours:**
- 120 parts/hour × 7.5 hours = 900 parts

**Good parts (96% are good):**
- 100% - 4% defective = 96% good
- 900 × 0.96 = **864 good parts**

The machine makes **864 good parts** in a 7.5 hour shift.
----------------------------------------
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=70, output_tokens=281, output_tokens_details=OutputTokensDetails(thinking_tokens

## Keeping thinking in a multi-turn chat

In [6]:
# Save the FULL message so the signed thinking block is preserved
add_assistant_message(messages, response)
add_user_message(messages, "Now repeat it for a 10 hour shift.")

# Do not edit the thinking text, or the signature check will reject it
response2 = chat(messages, thinking=True)
print(text_from_message(response2))

# Good Parts Calculation (10 Hour Shift)

**Total parts made in 10 hours:**
- 120 parts/hour × 10 hours = 1,200 parts

**Good parts (96% are good):**
- 1,200 × 0.96 = **1,152 good parts**

The machine makes **1,152 good parts** in a 10 hour shift.
